# Wrangling Harmonic-EEG Data

This notebook gets the Harmonic-EEG dataset ready for analysis. The dataset has EEG recordings from 40 musicians, either classical musicians or jazz-trained improvisers, while they listened to three-chord progressions where the middle chord was occasionally changed. Before analyzing the EEG, we check who the participants are and that every participant's files are present and complete.

### Import libraries

Load the Python libraries used in this notebook: `pandas` for working with tables, `pathlib` for finding files in the `Data` folder, and `json` for reading the dataset's `.json` metadata files.

In [13]:
import pandas as pd
from pathlib import Path
import json

### Load and summarize participant information

`participants.tsv` has one row per participant with their ID, group (classical or improviser), handedness, age and gender. The cells below load it and summarize who is in the study:
- how many participants there are in total and in each group
- the average age and age range
- how many of each gender and handedness are in each group

In [10]:
# Load the TSV file into a DataFrame
participants_data = pd.read_csv('Data/participants.tsv', sep='\t')

# Convert to dataframe
df = pd.DataFrame(participants_data)

# Display the first few rows
df.head()

,participant_id,group,handedness,age,gender
0,sub-001,Cl,R,26,M
1,sub-002,Cl,R,26,M
2,sub-003,Cl,R,42,F
3,sub-004,Cl,R,24,F
4,sub-005,Imp,L,27,M


In [6]:
# Full names for the group codes used in the TSV
group_names = {"Cl": "classical musicians who don't improvise",
               "Imp": "improvisers (jazz-trained)"}

# Number of unique participants (participant_id is the primary key)
print(f"Participants: {df['participant_id'].nunique()}")

# Number of participants in each group
print("Group:")
for g, n in df["group"].value_counts().items():
    print(f"  {g} ({group_names[g]}): {n}")

# Mean and range of age
print(f"Age: mean {df['age'].mean():.1f}, range {df['age'].min()}–{df['age'].max()}")

# Number of participants of each gender
print("Gender:", df["gender"].value_counts().to_dict())

# Number of participants of each handedness
print("Handedness:", df["handedness"].value_counts().to_dict())

Participants: 40
Group:
  Imp (improvisers (jazz-trained)): 24
  Cl (classical musicians who don't improvise): 16
Age: mean 25.3, range 18–42
Gender: {'M': 24, 'F': 16}
Handedness: {'R': 31, 'A': 5, 'L': 4}


In [9]:
# Summary statistics for age
df["age"].agg(["count", "mean", "std", "min", "max"])

count    40.000000
mean     25.275000
std       5.514491
min      18.000000
max      42.000000
Name: age, dtype: float64

In [8]:
# Number of participants of each gender in each group
display(pd.crosstab(df["group"], df["gender"], margins=True))

# Number of participants of each handedness in each group
display(pd.crosstab(df["group"], df["handedness"], margins=True))

gender,F,M,All
group,,,
Cl,12,4,16
Imp,4,20,24
All,16,24,40


handedness,A,L,R,All
group,,,,
Cl,1,1,14,16
Imp,4,3,17,24
All,5,4,31,40


### Check that every participant has all their files

Each participant's `eeg` folder should contain the same set of files: the EEG recording itself (split into a `.set` header and an `.fdt` data file) plus smaller files describing the channels, electrodes and events. This cell lists every file with its size, then reports:
- any participant who is missing a file
- any EEG recording under 1 MB, which usually means it was never fully downloaded

In [11]:
root = Path("Data")

# List every file in each participant's eeg folder
rows = []
for file in sorted(root.glob("sub-*/eeg/*")):
    rows.append({
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "sub-001"
        "participant": file.name.split("_")[0],
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "events.tsv"
        "file": file.name.split("_")[-1],
        # Size in MB (0 if the file is a link to data that hasn't been downloaded)
        "size_MB": round(file.stat().st_size / 1e6, 2) if file.exists() else 0,
    })
files = pd.DataFrame(rows)

# Table with one row per participant and one column per file type (NaN = missing)
sizes = files.pivot(index="participant", columns="file", values="size_MB")

# Number of participants found
print(len(sizes), "participants found")

# Number of participants that have each file type (should all match the number above)
print("\nFiles per type:")
print(sizes.notna().sum())

# Participants missing at least one file type
missing = sizes[sizes.isna().any(axis=1)]
print("\nParticipants with missing files:")
print(missing if len(missing) else "  none")

# EEG recordings under 1 MB weren't fully downloaded
small = sizes.loc[(sizes["eeg.set"] < 1) | (sizes["eeg.fdt"] < 1), ["eeg.set", "eeg.fdt"]]
print("\nEEG recordings under 1 MB:")
print(small if len(small) else "  none")

40 participants found

Files per type:
file
channels.tsv        40
coordsystem.json    40
eeg.fdt             40
eeg.json            40
eeg.set             40
electrodes.tsv      40
events.json         40
events.tsv          40
dtype: int64

Participants with missing files:
  none

EEG recordings under 1 MB:
  none


In [ ]:
""" 
Go through each participant and inspect the channels.tsv for each participant to
check for missing data, to produce one table for the channels.tsv for each participant
"""

_IncompleteInputError: incomplete input (2682125433.py, line 1)